# C-18 Explainability Analysis

**Control:** C-18 Explainability Analysis, MAK-AI Responsible AI Toolkit. This notebook also carries the robustness part of the framework's coverage gap on testing a model's sensitivity to noise and small changes.

**Learning goal:** explain a model for the whole population (permutation importance and a SHAP summary) and for one case (a SHAP waterfall); find a spurious feature; test whether the explanation is stable and whether the model reacts to noise and small changes; and say plainly what explanations cannot show.

**Time to run:** about 3 minutes on an ordinary laptop. No GPU.

**Licence:** this notebook is Apache-2.0.

**Example data:** Statlog (German Credit Data) (Hofmann, 1994), DOI 10.24432/C5NC77, licence CC BY 4.0, 1,000 loan applications, from https://archive.ics.uci.edu/dataset/144/statlog+german+credit+data. It is credited here because CC BY 4.0 asks for attribution.

**A caveat before you start:** these are 1,000 credit applications from one German bank, collected in the 1970s and 1980s and released in 1994. The file combines sex with marital status in one code (`personal_status_sex`), which is a lesson in measurement: you cannot ask the data about sex alone. With 1,000 rows the explanations are noisy, and part of the notebook measures how noisy. One column in the notebook is planted on purpose and is labelled as synthetic where it appears.

## Linked risk areas and questions

Generated from content 5.0.0. Do not edit by hand.

Risk areas: 4

- IP-03: Opacity & Unexplainability
- IP-04: Metric Selection Bias
- IP-05: Model Robustness Failure
- IP-06: Reproducibility Failure

Questions: 20: Q-IP-01, Q-IP-03, Q-IP-07, Q-IP-08, Q-IP-16, Q-IP-17, Q-IP-18, Q-IP-09, Q-IP-10, Q-IP-12, Q-IP-13, Q-IP-15, Q-IP-20, Q-IP-21, Q-IP-22, Q-IP-23, Q-IP-24, Q-IP-EXTRA-01, Q-IP-EXTRA-02, Q-PO-11

## When to use this notebook

Use it when you are answering a question about whether you can explain what the model relies on and your status is **Not yet** or **Partly**, especially with the reason code `ny_needs_skills` ("Needs skills or guidance the team does not have yet"). It helps with:

- `Q-IP-07`, `Q-IP-17` and `Q-IP-23`: can you explain the model's behaviour, is it stable, and has it been tested against small changes?
- `Q-IP-01` and `Q-IP-08`: do you know which inputs the model relies on, and are any of them ones it should not use?
- `Q-IP-22`: can the people affected get an explanation they can understand?

You need a table with one row per case, the true outcome, and the columns the model reads.

In [ ]:
import hashlib
import importlib.metadata
import io
import platform
import urllib.request
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import PartialDependenceDisplay, permutation_importance
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

SEED = 20261030  # the one random seed; every random step below uses it
rng = np.random.default_rng(SEED)
pd.set_option('display.width', 150)
pd.set_option('display.max_columns', 30)

PACKAGES = ['pandas', 'numpy', 'scikit-learn', 'shap', 'matplotlib']

print('Python', platform.python_version())
for _package in PACKAGES:
    print(_package, importlib.metadata.version(_package))

In [ ]:
DATA_URL = 'https://archive.ics.uci.edu/static/public/144/data.csv'
DATA_SHA256 = 'f985b42636c9e28ce4fb3913738d1ebfd0f7947fda8d4406fe2ded5d84295acb'
DATA_FILE = 'uci-german-credit-144-data.csv'  # the cache file name is fixed here

CACHE_DIR = Path.home() / '.cache' / 'makrai-notebooks'


def fetch_checked(url, sha256, file_name):
    """Return the bytes of a dataset whose SHA-256 matches `sha256`.

    The file is downloaded once into the cache folder, and written there only after its hash
    matches. Every run, including the first, hashes the bytes before they are used, and a
    mismatch stops the notebook. `file_name` is fixed in this notebook, never taken from the address.
    """
    if not url.startswith('https://'):
        raise ValueError(f'the dataset address must be https: {url}')
    path = CACHE_DIR / file_name
    if path.exists():
        data, source, remedy = path.read_bytes(), f'the cached file {path}', f'Delete {path} and run again.'
    else:
        request = urllib.request.Request(url, headers={'User-Agent': 'makrai-notebooks'})
        try:
            with urllib.request.urlopen(request, timeout=120) as response:
                if not response.geturl().startswith('https://'):
                    raise RuntimeError(f'{url} redirected to a non-https address: {response.geturl()}')
                data = response.read()
        except OSError as error:
            raise RuntimeError(f'could not download {url}: {error}. If you cannot reach it, save the file by hand '
                               f'as {path}; it is checked against the pinned SHA-256 either way.') from error
        source, remedy = url, 'Do not use this download; check the dataset page for a new version.'
    found = hashlib.sha256(data).hexdigest()
    if found != sha256:
        raise RuntimeError(f'{file_name}: the SHA-256 of {source} is {found}, expected {sha256}. {remedy}')
    if not path.exists():
        CACHE_DIR.mkdir(parents=True, exist_ok=True)
        partial = path.with_name(path.name + '.part')
        partial.write_bytes(data)
        partial.replace(path)  # the cache never holds a half-written file
    return data


# The file has no column names and codes such as A11. Names and meanings are from the dataset's own description.
COLUMNS = ['status_checking', 'duration_months', 'credit_history', 'purpose', 'credit_amount', 'savings',
           'employment_since', 'instalment_rate', 'personal_status_sex', 'other_debtors', 'residence_since',
           'property', 'age', 'other_instalment_plans', 'housing', 'existing_credits', 'job', 'people_liable',
           'telephone', 'foreign_worker']
MEANING = {
    'A11': 'under 0 DM', 'A12': '0 to 200 DM', 'A13': '200 DM or more', 'A14': 'no checking account',
    'A30': 'no credits or all paid back', 'A31': 'all paid back at this bank', 'A32': 'paid back duly so far',
    'A33': 'delay in the past', 'A34': 'critical account or other credits',
    'A40': 'car (new)', 'A41': 'car (used)', 'A42': 'furniture or equipment', 'A43': 'radio or television',
    'A44': 'domestic appliances', 'A45': 'repairs', 'A46': 'education', 'A48': 'retraining', 'A49': 'business',
    'A410': 'other purpose',
    'A61': 'under 100 DM', 'A62': '100 to 500 DM', 'A63': '500 to 1000 DM', 'A64': '1000 DM or more', 'A65': 'unknown or none',
    'A71': 'unemployed', 'A72': 'under 1 year', 'A73': '1 to 4 years', 'A74': '4 to 7 years', 'A75': '7 years or more',
    'A91': 'male, divorced or separated', 'A92': 'female, divorced, separated or married', 'A93': 'male, single',
    'A94': 'male, married or widowed', 'A95': 'female, single',
    'A101': 'none', 'A102': 'co-applicant', 'A103': 'guarantor',
    'A121': 'real estate', 'A122': 'savings agreement or life insurance', 'A123': 'car or other', 'A124': 'unknown or none',
    'A141': 'bank', 'A142': 'stores', 'A143': 'none',
    'A151': 'rent', 'A152': 'own', 'A153': 'for free',
    'A171': 'unemployed or unskilled, non-resident', 'A172': 'unskilled, resident', 'A173': 'skilled employee',
    'A174': 'management or highly qualified',
    'A191': 'none', 'A192': 'yes', 'A201': 'yes', 'A202': 'no',
}

# To use your own file, replace the next lines with:  raw = Path('your-file.csv').read_bytes()  and  df = pd.read_csv(io.BytesIO(raw))
raw = fetch_checked(DATA_URL, DATA_SHA256, DATA_FILE)
DATA_SHA256_USED = hashlib.sha256(raw).hexdigest()  # the evidence summary records the hash of the bytes used
df = pd.read_csv(io.BytesIO(raw))
df.columns = COLUMNS + ['class']  # Adapt step: delete these decoding lines for your own data
for _column in df.columns[:-1]:
    if df[_column].dtype == object or str(df[_column].dtype).startswith('str'):
        df[_column] = df[_column].map(MEANING)
print(f'{len(df):,} rows, {df.shape[1]} columns; SHA-256 of the data used: {DATA_SHA256_USED[:12]}...')

In [ ]:
# ADAPT HERE. Edit only this cell to run the notebook on your own data. Every later cell reads only these names.
LABEL = 'class'                      # the column that holds the true outcome
BAD_OUTCOME = 2                      # the value of LABEL the model predicts (in this file 2 means a bad credit risk)
OUTCOME_WORDS = 'the loan not being repaid'    # the same thing in plain words, used in the explanation for end users
NUMERIC = ['duration_months', 'credit_amount', 'instalment_rate', 'residence_since', 'age',
           'existing_credits', 'people_liable']
CATEGORICAL = ['status_checking', 'credit_history', 'purpose', 'savings', 'employment_since',
               'personal_status_sex', 'other_debtors', 'property', 'other_instalment_plans', 'housing', 'job',
               'telephone', 'foreign_worker']
CASE_INDEX = 0                       # which person in the test set to explain on their own (0 is the first)
PLANTED_LEAK = 'planted_leak'        # name of the synthetic leaky column added in Step 5; None skips Step 5
SECOND_SEED = 7                      # a second seed, used to check the explanation is stable
TOP_FEATURES = 5                     # how many top features to compare
N_REPEATS = 10                       # how many times each column is shuffled for permutation importance
NOISE_LEVELS = [0.05, 0.10, 0.25]    # noise added to numeric columns, as a share of each column's spread
SMALL_CHANGE = 0.10                  # the small change in Step 8, as a share of a column's spread

### Step 1: the model to be explained

**What this does.** It splits the cases into a training part (70 percent) and a test part (30 percent), turns the categorical columns into yes/no indicator columns, and fits a gradient-boosted tree model (`HistGradientBoostingClassifier`) that predicts the bad outcome. Scores below are on the test part only.

**What it shows.** That there is a model with some skill. If it is no better than guessing, there is nothing to explain.

**What it does not show.** That the model is good enough to use, fair, or right. Explanations describe a model whatever its quality.

In [ ]:
def split_and_fit(data, columns_numeric, columns_categorical, seed):
    """Fit the model on 70 percent of the cases and return it with the test part."""
    target = (data[LABEL] == BAD_OUTCOME).astype(int)
    train_i, test_i = train_test_split(data.index, test_size=0.3, stratify=target, random_state=seed)
    preprocess = ColumnTransformer(
        [('categories', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), columns_categorical)],
        remainder='passthrough', verbose_feature_names_out=False).set_output(transform='pandas')
    pipe = Pipeline([('prepare', preprocess), ('model', HistGradientBoostingClassifier(random_state=seed))])
    inputs = data[columns_numeric + columns_categorical].copy()
    inputs[columns_numeric] = inputs[columns_numeric].astype(float)  # partial dependence needs floating point columns
    pipe.fit(inputs.loc[train_i], target[train_i])
    return pipe, inputs.loc[train_i], inputs.loc[test_i], target[train_i], target[test_i]


pipe, X_train, X_test, y_train, y_test = split_and_fit(df, NUMERIC, CATEGORICAL, SEED)
probability = pipe.predict_proba(X_test)[:, 1]
print(f'training cases {len(X_train)}, test cases {len(X_test)}, share with the bad outcome {y_test.mean():.2f}')
print(f'AUC on the test cases: {roc_auc_score(y_test, probability):.3f}')
print(f'accuracy at 0.5: {accuracy_score(y_test, probability >= 0.5):.3f} (always saying "good" would score {1 - y_test.mean():.3f})')

### Step 2: which columns does the model rely on? Permutation importance

**What this does.** On the test cases, it shuffles one column at a time (so that column no longer says anything true about the case) and measures how much the model's AUC falls. A big fall means the model leans on that column. It repeats the shuffle `N_REPEATS` times and shows the spread.

**What it shows.** A global ranking of the columns the model needs, measured on cases it has not seen.

**What it does not show.** That a column is a cause, or that it is acceptable to use. If two columns say nearly the same thing, shuffling one of them costs little because the other covers for it, so each can look unimportant while together they matter. And a column with an importance near zero may still matter for a few people.

In [ ]:
result = permutation_importance(pipe, X_test, y_test, scoring='roc_auc', n_repeats=N_REPEATS, random_state=SEED)
importance = pd.DataFrame({'column': X_test.columns, 'importance': result.importances_mean, 'spread': result.importances_std})
importance = importance.sort_values('importance', ascending=False).reset_index(drop=True)
print(importance.round(4).head(10).to_string(index=False))

fig_importance, axis = plt.subplots(figsize=(7, 4.5))
top = importance.head(10).iloc[::-1]
axis.barh(top['column'], top['importance'], xerr=top['spread'])
axis.set_xlabel('fall in AUC when the column is shuffled')
axis.set_title('Permutation importance on the test cases')
fig_importance.tight_layout()

### Step 3: how do the columns push the score? SHAP summary

**What this does.** SHAP gives every case a number for every column: how far that column pushed this case's score up or down compared with an average case. The numbers for one case add up to the model's score for it (here in log-odds, a scale where positive means toward the bad outcome). The first picture ranks the columns by their average size. The second plots every case as a dot: its position left or right is the push, and its colour is whether the column's value is high or low. Columns that came from a category appear as indicator columns, one per category.

**What it shows.** Which columns matter and in which direction, across the whole test set.

**What it does not show.** Cause. SHAP explains the model, not the world: it says what the model does with a column, not what would happen to a person if the column changed. When columns are related, SHAP shares the credit among them in ways that depend on the method's assumptions.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter('ignore')  # the tree explainer prints a version note for some models; the numbers are unaffected
    prepared_train = pipe.named_steps['prepare'].transform(X_train)
    prepared_test = pipe.named_steps['prepare'].transform(X_test)
    explainer = shap.TreeExplainer(pipe.named_steps['model'], data=prepared_train.sample(100, random_state=SEED))
    explanation = explainer(prepared_test)

plt.figure()
shap.plots.bar(explanation, max_display=10, show=False)
fig_shap_bar = plt.gcf()
plt.figure()
shap.plots.beeswarm(explanation, max_display=10, show=False)
fig_shap_beeswarm = plt.gcf()

### Step 4: why this person? A SHAP waterfall for one case

**What this does.** It takes the one case you chose in `CASE_INDEX` and draws how each column moved the score from the average to this case's score, largest first. The picture stays on the screen of the person running the notebook, and is not saved to the evidence folder, because it describes one person's record.

**What it shows.** A reason, in the model's terms, for one decision.

**What it does not show.** A reason the person could act on. "The model gave weight to the length of the loan" is not advice, and a different, equally accurate model could give different reasons for the same decision.

In [ ]:
plt.figure()
shap.plots.waterfall(explanation[CASE_INDEX], max_display=10, show=False)
fig_waterfall = plt.gcf()
print(f'model score for this case: {probability[CASE_INDEX]:.2f} (probability of the bad outcome)')

### Step 5: partial dependence for the top columns

**What this does.** For each of the top numeric columns, it varies that column across its range for every test case, keeps the others as they are, and averages the model's score. The curve shows the model's average response.

**What it shows.** The shape of the relationship the model learned: rising, falling, flat, or jumping at one value.

**What it does not show.** What the real relationship is. Where the data is thin, the curve is unreliable (the marks along the bottom show where the cases are). Averaging can also hide groups for whom the response goes the other way.

In [ ]:
numeric_top = [c for c in importance['column'] if c in NUMERIC][:3]
display = PartialDependenceDisplay.from_estimator(pipe, X_test, numeric_top, kind='average', random_state=SEED)
fig_pdp = display.figure_
fig_pdp.tight_layout()

### Step 6: a spurious feature (a planted leak)

**What this does.** It adds one column that is **synthetic and planted on purpose**: the true outcome plus random noise. Nothing like it would exist at the time of a real prediction (it is a stand-in for something recorded after the fact, such as a collections note). It refits the model with the extra column and compares the importance and the score with Step 2.

**What it shows.** What a leak looks like: one column leaps to the top and the model becomes almost too good. In real work, this is the pattern to look for when a result seems too good, or when a top column is something that could not have been known when the decision was made.

**What it does not show.** How to find a leak you cannot name. The check is a person asking of each top column, "would we have known this at prediction time?"

In [ ]:
if PLANTED_LEAK is not None:
    leaky = df.copy()
    leaky[PLANTED_LEAK] = (leaky[LABEL] == BAD_OUTCOME).astype(float) + rng.normal(0, 0.5, len(leaky))  # SYNTHETIC
    pipe_leak, Xl_train, Xl_test, yl_train, yl_test = split_and_fit(leaky, NUMERIC + [PLANTED_LEAK], CATEGORICAL, SEED)
    leak_result = permutation_importance(pipe_leak, Xl_test, yl_test, scoring='roc_auc', n_repeats=N_REPEATS, random_state=SEED)
    leak_importance = pd.DataFrame({'column': Xl_test.columns, 'importance': leak_result.importances_mean})
    leak_importance = leak_importance.sort_values('importance', ascending=False).reset_index(drop=True)
    print(f'AUC without the planted column {roc_auc_score(y_test, probability):.3f}, with it {roc_auc_score(yl_test, pipe_leak.predict_proba(Xl_test)[:, 1]):.3f}')
    print(leak_importance.head(6).round(4).to_string(index=False))
else:
    leak_importance = pd.DataFrame(columns=['column', 'importance'])
    print('Step 6 skipped: no planted column was named in the Adapt cell.')

### Step 7: is the explanation stable?

**What this does.** It repeats the split, the fit and the permutation importance with a second seed and compares the two rankings: how many of the top columns they share, and how well the whole ranking agrees (Spearman rank correlation, where 1 is identical order and 0 is unrelated).

**What it shows.** Whether the story "the model relies on these columns" depends on which cases happened to land in the test part.

**What it does not show.** Stability across different data, a different period, or a different model. Two seeds on one dataset is the smallest possible check.

In [ ]:
pipe_b, Xb_train, Xb_test, yb_train, yb_test = split_and_fit(df, NUMERIC, CATEGORICAL, SECOND_SEED)
result_b = permutation_importance(pipe_b, Xb_test, yb_test, scoring='roc_auc', n_repeats=N_REPEATS, random_state=SECOND_SEED)
importance_b = pd.Series(result_b.importances_mean, index=Xb_test.columns)
importance_a = pd.Series(result.importances_mean, index=X_test.columns)
shared = set(importance_a.nlargest(TOP_FEATURES).index) & set(importance_b.nlargest(TOP_FEATURES).index)
rank_agreement = importance_a.rank().corr(importance_b.rank())
stability = pd.DataFrame({'seed': [SEED, SECOND_SEED], 'AUC': [roc_auc_score(y_test, probability),
                          roc_auc_score(yb_test, pipe_b.predict_proba(Xb_test)[:, 1])]})
print(stability.round(3).to_string(index=False))
print(f'top {TOP_FEATURES} columns shared by the two runs: {len(shared)} of {TOP_FEATURES}: {sorted(shared)}')
print(f'rank agreement (Spearman) over all columns: {rank_agreement:.2f}')

### Step 8: noise and small changes (the robustness check)

**What this does.** Two tests of how steady the model is.

1. **Noise.** For each level in `NOISE_LEVELS`, it adds random noise to every numeric column of the test cases (the noise has a size equal to that share of the column's spread) and measures how many predictions cross the 0.5 line and how much the AUC changes.
2. **Small changes.** For each numeric column in turn, it raises the column by `SMALL_CHANGE` of its spread for every case and measures the average change in the predicted probability.

**What it shows.** Whether tiny differences in the input, such as a rounding error or a typing slip, can flip a decision, and which columns the model is most sensitive to.

**What it does not show.** Robustness against someone trying to fool the model on purpose (adversarial testing is a different exercise), or against changes in the world (see C-21).

In [ ]:
spread = X_test[NUMERIC].std()
baseline_decision = probability >= 0.5
noise_rows = []
for level in NOISE_LEVELS:
    noisy = X_test.copy()
    noisy[NUMERIC] = noisy[NUMERIC] + rng.normal(0, 1, noisy[NUMERIC].shape) * spread.to_numpy() * level
    noisy_probability = pipe.predict_proba(noisy)[:, 1]
    noise_rows.append({'noise (share of spread)': level,
                       'decisions that flip': float(np.mean((noisy_probability >= 0.5) != baseline_decision)),
                       'AUC change': roc_auc_score(y_test, noisy_probability) - roc_auc_score(y_test, probability)})
noise_table = pd.DataFrame(noise_rows)

change_rows = []
for column in NUMERIC:
    changed = X_test.copy()
    changed[column] = changed[column] + SMALL_CHANGE * spread[column]
    change_rows.append({'column raised': column, 'average change in probability': float(np.mean(pipe.predict_proba(changed)[:, 1] - probability))})
change_table = pd.DataFrame(change_rows).sort_values('average change in probability', key=abs, ascending=False)
print(noise_table.round(3).to_string(index=False))
print()
print(change_table.round(4).to_string(index=False))

### Step 9: an explanation for the person affected

**What this does.** It fills a plain-language template from the SHAP values of the case you chose, naming the details that raised and lowered the estimate most. It is a starting point to adapt and to test with real readers, not finished wording. The result is shown here only and is not saved, because it describes one person.

**What it shows.** How a technical explanation can become something a person might read.

**What it does not show.** That the person understood it, that it is complete, or that it is a fair reason. The template says the estimate is not a decision and that a person can ask for a review. Say that only if it is true in your process.

In [ ]:
TEMPLATE = (
    'The model estimates a {chance:.0%} chance of {outcome}. This is an estimate based on past cases, not a decision about you. '
    'The details that raised the estimate most were: {up}. The details that lowered it most were: {down}. '
    'The model can only use the details listed here. If any of them is wrong, tell us. You can ask for a person to review this estimate.'
)


def explain_in_words(row, chance, how_many=3):
    order = np.argsort(row.values)
    up = [row.feature_names[i].replace('_', ' ') for i in order[::-1][:how_many] if row.values[i] > 0]
    down = [row.feature_names[i].replace('_', ' ') for i in order[:how_many] if row.values[i] < 0]
    return TEMPLATE.format(chance=chance, outcome=OUTCOME_WORDS, up=', '.join(up) or 'none', down=', '.join(down) or 'none')


print(explain_in_words(explanation[CASE_INDEX], probability[CASE_INDEX]))

### Stop and decide: do the columns the model relies on make sense?

The notebook shows what the model relies on. Whether that is acceptable is a judgment for people who know the setting.

Questions to ask in the room, with a domain expert who did not build the model:

1. Look at the top columns in Steps 2 and 3. Would an experienced person in this field expect them to matter? Which would surprise them?
2. Is any top column one the model should not use, or one that stands in for something it should not (sex, age, nationality, religion, health)? Here `personal_status_sex` and `foreign_worker` are candidates: what would you do if they are important?
3. Would each top column have been known at the moment of the decision? If not, it is a leak.
4. Is the story stable (Step 7)? If the top columns change with the seed, what you tell people about the model may be an accident.
5. How much can noise and small changes move a decision (Step 8)? Is that acceptable for the people who would be on the wrong side of it?
6. Could the person affected understand and act on the explanation (Step 9)? Who checks that with real readers?

Write your choices in the next cell. Leave it blank if you have not decided, and the summary will say so.

In [ ]:
# YOUR CHOICES. The notebook does not fill this in. Plain words are fine.
COLUMNS_THAT_MAKE_SENSE = ''     # the top columns an expert accepted, and why
COLUMNS_THAT_DO_NOT = ''         # columns that surprised the expert or should not be used, and what you will do
LEAK_CHECK = ''                  # for each top column: would it be known at decision time?
ROBUSTNESS_VERDICT = ''          # is the sensitivity in Step 8 acceptable, and for whom
WHO_REVIEWED = ''                # the role of the domain expert who reviewed this, not a name

In [ ]:
EVIDENCE_DIR = Path('evidence')
EVIDENCE_DIR.mkdir(exist_ok=True)

# Numbers and pictures about the model as a whole. The single-case waterfall and wording are not saved.
importance.round(4).to_csv(EVIDENCE_DIR / 'C-18_permutation_importance.csv', index=False)
leak_importance.round(4).to_csv(EVIDENCE_DIR / 'C-18_planted_leak_importance.csv', index=False)
stability.round(4).to_csv(EVIDENCE_DIR / 'C-18_seed_stability.csv', index=False)
noise_table.round(4).to_csv(EVIDENCE_DIR / 'C-18_noise_sensitivity.csv', index=False)
change_table.round(4).to_csv(EVIDENCE_DIR / 'C-18_small_change_sensitivity.csv', index=False)
fig_importance.savefig(EVIDENCE_DIR / 'C-18_permutation_importance.png', dpi=100, bbox_inches='tight')
fig_shap_bar.savefig(EVIDENCE_DIR / 'C-18_shap_bar.png', dpi=100, bbox_inches='tight')
fig_shap_beeswarm.savefig(EVIDENCE_DIR / 'C-18_shap_summary.png', dpi=100, bbox_inches='tight')
fig_pdp.savefig(EVIDENCE_DIR / 'C-18_partial_dependence.png', dpi=100, bbox_inches='tight')

versions = '\n'.join(f'- {package} {importlib.metadata.version(package)}' for package in PACKAGES)
summary = f"""# C-18 Explainability Analysis: evidence summary

Attach this file, with the CSV and PNG files beside it, to the decision it supports (questions Q-IP-01, Q-IP-07, Q-IP-08, Q-IP-17, Q-IP-22, Q-IP-23). It holds numbers and pictures about the model as a whole, never the data and never one person's record.

## What was run
- Outcome predicted: {LABEL} equal to {BAD_OUTCOME}; {len(X_train) + len(X_test):,} cases, {len(X_test):,} in the test set
- Model: gradient-boosted trees; AUC on the test set {roc_auc_score(y_test, probability):.3f}
- Columns used: {', '.join(NUMERIC + CATEGORICAL)}
- Seeds: {SEED} and {SECOND_SEED}

## Top columns by permutation importance
{importance.head(TOP_FEATURES).round(4).to_string(index=False)}

## Stability across two seeds
- Top {TOP_FEATURES} columns shared: {len(shared)} of {TOP_FEATURES}
- Rank agreement over all columns (Spearman): {rank_agreement:.2f}

## Robustness
{noise_table.round(3).to_string(index=False)}

## Choices made in the stop-and-decide cell
- Columns that make sense: {COLUMNS_THAT_MAKE_SENSE or 'not written yet'}
- Columns that do not, and what we will do: {COLUMNS_THAT_DO_NOT or 'not written yet'}
- Leak check: {LEAK_CHECK or 'not written yet'}
- Robustness verdict: {ROBUSTNESS_VERDICT or 'not written yet'}
- Reviewed by (role): {WHO_REVIEWED or 'not written yet'}

## Files beside this summary
C-18_permutation_importance.csv, C-18_planted_leak_importance.csv, C-18_seed_stability.csv, C-18_noise_sensitivity.csv, C-18_small_change_sensitivity.csv and four PNG files (permutation importance, SHAP bar, SHAP summary, partial dependence). The planted leak is a synthetic column added to show what a leak looks like; it is not part of the model being assessed.

## Package versions
- Python {platform.python_version()}
{versions}

## Data
SHA-256 of the data file used: {DATA_SHA256_USED}
"""
(EVIDENCE_DIR / 'C-18_summary.md').write_text(summary, encoding='utf-8')
print(summary)

## What this does not show, and where to go next

- **An explanation is a description of a model, not a justification.** Two models with the same accuracy can give different reasons for the same case. A reason from SHAP or permutation importance is not a reason the decision is right or fair.
- **Importance is not cause, and not fairness.** A column can matter because it stands in for something the model should not use. The group metrics in C-13 and C-14 are the next step if a top column is sensitive.
- **The explanations here are noisy.** With 1,000 rows, rankings move with the seed, which Step 7 measures. A larger or different dataset will tell a different story.
- **Robustness here is mild.** Random noise and a small shift are the smallest tests. They say nothing about a motivated person trying to game the model, or about the world changing (see C-21).
- **End users are not the audience of a SHAP chart.** Step 9 is a template to test with real readers, not a finished explanation.

Where to go next: C-13 Subgroup Fairness Metrics and C-14 Disaggregated Performance Analysis (does the model behave differently for groups), C-21 Monitoring and Drift Detection (does it keep behaving), and the model card template, whose section on interpretability and intended use takes the findings of this notebook.